### Importing Libraries

In [120]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OrdinalEncoder



##  Play Tennis Dataset — Categorical Features

In this step, we load the Play Tennis dataset, our first dataset for testing the from-scratch decision tree implementation.

```python
df = pd.read_csv("./play_tennis_dataset.csv")
```

### About the Dataset

The Play Tennis dataset contains categorical features such as `Outlook`, `Temperature`, `Humidity`, and `Wind`. The target variable, `Play Tennis`, indicates whether tennis should be played (`Yes` or `No`).

### Purpose

- Test our decision tree on a dataset containing only categorical features.
- Understand how the tree creates branches for different feature values.
- Verify the implementation's predictions against Scikit-learn.

This dataset provides a simple starting point before testing our decision tree on datasets containing numerical and mixed-type features.### Data Preprocessing

In [121]:
df = pd.read_csv("./play_tennis_dataset.csv")


In [122]:
df.head(10)

,Day,Outlook,Temperature,Humidity,Wind,Play
0,D1,Overcast,Mild,Normal,Strong,Yes
1,D2,Sunny,Mild,Normal,Strong,Yes
2,D3,NaN,Mild,High,Strong,No
3,D4,Sunny,Mild,High,Weak,Yes
4,D5,Sunny,Cool,Normal,Strong,Yes
5,D6,NaN,Cool,Normal,Strong,Yes
6,D7,Sunny,Mild,High,Weak,Yes
7,D8,Sunny,Mild,Normal,Weak,Yes
8,D9,Rainy,Mild,Normal,Weak,No
9,D10,Rainy,Cool,High,Weak,No


In [123]:
df.isna().sum()

Day              0
Outlook        399
Temperature    333
Humidity       233
Wind           366
Play             0
dtype: int64

In [124]:
df.dropna(inplace=True,ignore_index=True)

In [125]:
X = df.drop(columns=["Day","Play"]).to_numpy()
y = df["Play"].to_numpy()
feature_names = df.drop(columns=["Day","Play"]).columns.to_list()

### Spliiting Data into Train Test

In [126]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,train_size=0.8)

In [127]:
print("shape of X train",X_train.shape)
print("shape of Y train",y_train.shape)
print("shape of X_test",X_test.shape)
print("shape of y_test",y_test.shape)

shape of X train (4340, 4)
shape of Y train (4340,)
shape of X_test (1085, 4)
shape of y_test (1085,)


In [128]:
X_train[:5]

array([['Rainy', 'Hot', 'High', 'Weak'],
       ['Sunny', 'Hot', 'Normal', 'Weak'],
       ['Sunny', 'Cool', 'High', 'Weak'],
       ['Sunny', 'Mild', 'Normal', 'Weak'],
       ['Overcast', 'Mild', 'Normal', 'Strong']], dtype=object)

##  Entropy Calculation

Entropy measures the impurity or uncertainty in a set of target labels. In a decision tree, it helps us determine how mixed the classes are at a particular node.

The entropy formula is:

$$
H(Y) = -\sum_{i=1}^{C} p_i \log_2(p_i)
$$

Where:

- \(C\) is the number of unique classes.
- \(p_i\) is the proportion of samples belonging to class \(i\).

**Interpretation:**
- **Entropy = 0:** The node is pure; all samples belong to one class.
- **Higher entropy:** The node contains a greater mixture of classes.
- **Maximum entropy:** The classes are equally distributed.

The function below calculates entropy from scratch using NumPy. It returns zero for an empty input or a node containing only one class.

In [129]:
def compute_entropy(y):
    entropy = 0
    # 1. Find the unique classes and their counts
    if len(y)!=0:
        n_unique,unique_counts = np.unique(y,return_counts=True)

        if len(n_unique)>1:
            # 2. Calculate  the probability of each class
            prob = unique_counts/len(y)

            entropy = -(np.sum(prob*np.log2(prob)))
    return entropy


In [130]:
y = ['Yes','No']
compute_entropy(y)

np.float64(1.0)

In [131]:
print("Entropy at root node: ", compute_entropy(y_train))

Entropy at root node:  0.9165191824508095


## Gini Impurity

Gini impurity is a measure of class impurity used by decision trees to evaluate the quality of a split. Like entropy, it measures how mixed the target classes are at a node.

The formula is:

$$
Gini(Y) = 1 - \sum_{i=1}^{C} p_i^2
$$

Here, \(p_i\) represents the proportion of samples belonging to class \(i\).

A Gini value of zero indicates a pure node, where all samples belong to the same class. Higher values indicate greater class impurity.

Gini impurity and entropy serve the same general purpose, but use different mathematical formulas. Gini is often computationally cheaper because it avoids logarithmic calculations.

In [132]:
def gini(y):
    gini = 0
    labels,counts = np.unique(y,return_counts=True)
    if len(labels)>1:

        prob = counts/len(y)
        gini =  1 - np.sum(prob**2)

    return gini


In [133]:
gini(["Yes","Yes","Yes","No","No"])

np.float64(0.48)

### Selecting the Impurity Criterion

The `calculate_impurity()` function allows the user to choose between **Gini Impurity** and **Entropy** for evaluating the quality of a decision tree split.

- **Gini:** Calls the `gini()` function to calculate Gini impurity.
- **Entropy:** Calls the `compute_entropy()` function to calculate entropy.

The `criterion` parameter determines which impurity measure is used throughout the decision tree implementation. This makes the algorithm flexible and allows us to compare the results of both criteria.

**Note:** The criterion must be either `"gini"` or `"entropy"`.

In [134]:
def calculate_impurity(y,criterion):
    if criterion == "gini":
        return gini(y)
    elif criterion == "entropy":
        return compute_entropy(y)
    else:
        raise ValueError("criterion must be either 'gini' or 'entropy'")


## Splitting the Dataset

The `split_dataset()` function divides the dataset according to a selected feature. The splitting strategy depends on whether the feature is categorical or numerical.

### Categorical Features

For a categorical feature, the function identifies all unique values and creates a separate subset for each category. For example, a `Sex` feature can produce separate branches for `male` and `female`.

### Numerical Features

For a numerical feature, the function:

1. Sorts the unique feature values.
2. Calculates candidate thresholds using the midpoints between adjacent unique values.
3. Divides the dataset into two subsets for each threshold:
   - **Left branch:** feature value less than or equal to the threshold.
   - **Right branch:** feature value greater than the threshold.
4. Calculates the information gain for each candidate threshold using entropy.
5. Retains the threshold that produces the highest information gain.

The function returns the categorical subsets or the best numerical threshold and its corresponding subsets. If a numerical feature contains fewer than two unique values, no valid threshold can be formed, so an empty dictionary is returned.

In [135]:
def split_dataset(X, y, feature,feature_type,criterion):
    #taking out specific column
    selected_feature_vals = X[:,feature]
    if feature_type == "categorical":
        n_unique= np.unique(selected_feature_vals)

        subsets = {}

        for val in n_unique:
            mask = selected_feature_vals == val
            X_subset = X[mask]
            y_subset = y[mask]
            labels,counts = np.unique(y_subset,return_counts=True)
            subsets[val] = (X_subset,y_subset,(labels,counts))
        return subsets
    elif feature_type == "numerical":
#         step 1 sort values and take  unique from them
        unique_val = np.sort(np.unique(selected_feature_vals))
        if len(unique_val)<2:
            return {}
        # this will take columns till last + will take except first to last add them and give mean of adjacent vals
        thresholds = (unique_val[:-1] + unique_val[1:])/2
        best_subsets = {}
        highest_gain = -np.inf

        for threshold in thresholds:
            left_mask= selected_feature_vals<=threshold
            right_mask = selected_feature_vals>threshold

            if not left_mask.any() or not right_mask.any():
                continue

            subsets = {
                "left" : (X[left_mask],y[left_mask]),
                "right" : (X[right_mask],y[right_mask])
            }

            weighted_entropy = sum(len(y_sub)/len(y) * calculate_impurity(y_sub,criterion)
                                   for x_sub,y_sub in subsets.values())

            gain =  calculate_impurity(y,criterion) - weighted_entropy

            if  gain > highest_gain:
                highest_gain = gain
                best_subsets = {"threshold" : threshold,
                                "subsets" : subsets}
        return best_subsets




##  Information Gain

Information Gain measures how much impurity decreases after splitting a dataset using a feature. It helps a decision tree select useful features for making decisions.

### Formula

**Information Gain = Parent Impurity − Weighted Child Impurity**

The weighted impurity of the child nodes is calculated as:

$$
\text{Weighted Child Impurity}
=
\sum_{j=1}^{k}
\left(
\frac{|Y_j|}{|Y|}
\times I(Y_j)
\right)
$$

Therefore, the complete formula is:

$$
\boxed{
IG = I(Y) -
\sum_{j=1}^{k}
\left(
\frac{|Y_j|}{|Y|}
\times I(Y_j)
\right)
}
$$

### Where:

- $IG$ = Information Gain.
- $I(Y)$ = Impurity of the parent node.
- $I(Y_j)$ = Impurity of the j-th child node.
- $|Y_j|$ = Number of samples in the j-th child node.
- $|Y|$ = Total number of samples in the parent node.
- $k$ = Total number of child nodes.

### How It Works

1. Calculate the impurity of the parent node.
2. Calculate the impurity of each child node.
3. Multiply each child's impurity by its proportion of the parent samples.
4. Add the weighted impurities of all child nodes.
5. Subtract the weighted child impurity from the parent impurity.

In this implementation, Gini impurity is used to evaluate both categorical and numerical splits. A higher information gain indicates a greater reduction in impurity and a more effective split.

In [136]:
def information_gain(X,y,feature,feature_type,criterion):
    weighted_entropy = 0
    subsets = split_dataset(X,y,feature,feature_type,criterion)
    if not subsets:
        return 0

    if feature_type == "categorical":
        parent_entropy = calculate_impurity(y,criterion)
        for val, (X_sub, y_sub,(n_unique,count)) in subsets.items():
            weighted_entropy += (len(y_sub)/len(y)) *calculate_impurity(y_sub,criterion)

    elif feature_type == "numerical":
        parent_entropy = calculate_impurity(y,criterion)
        best_split = subsets["subsets"]
        weighted_entropy = sum(len(y_sub)/len(y) * calculate_impurity(y_sub,criterion)
        for x_sub,y_sub in best_split.values())


    return parent_entropy - weighted_entropy






## Best Feature Selection

The `best_feature()` function selects the feature that provides the highest information gain among all available features.

### How It Works

1. Iterate through all features in the dataset.
2. Calculate the information gain of each feature using the `information_gain()` function.
3. Store each feature's information gain in a dictionary, where the key is the feature index and the value is its information gain.
4. Find and return the index of the feature with the highest information gain.

The selected feature is used to split the dataset at the current decision tree node.

**Note:** A higher information gain indicates a greater reduction in impurity, making that feature a better candidate for splitting.

In [137]:
def best_feature(X, y, feature_types,criterion):
    info_gain = {}

    if X.shape[1] != 0:
        for feature in range(X.shape[1]):
            gain = information_gain(X, y, feature, feature_types[feature],criterion)
            info_gain[feature] = gain

        max_key = max(info_gain, key=info_gain.get)
        return max_key

    return None

##  Feature Type Detection

The `detect_feature_types()` function identifies whether each feature in the dataset is numerical or categorical.

### How It Works

1. Initialize an empty list to store the feature types.
2. Iterate through each column of the feature matrix `X`.
3. Attempt to convert the column values to floating-point numbers using `astype(float)`.
4. If the conversion succeeds, classify the feature as **numerical**.
5. If a `ValueError` or `TypeError` occurs, classify the feature as **categorical**.
6. Return the list of detected feature types.

### Example

- `Age` → Numerical
- `Fare` → Numerical
- `Sex` → Categorical
- `Embarked` → Categorical

The detected feature types determine how the decision tree splits each feature: numerical features use threshold-based splits, while categorical features use category-based splits.

**Note:** Missing values and mixed data types may require preprocessing before feature type detection.

In [138]:
def detect_feature_types(X):
    feature_types = []

    for feature in range(X.shape[1]):
        column = X[:, feature]

        try:
            column = column.astype(float)
            feature_types.append("numerical")
        except (ValueError, TypeError):
            feature_types.append("categorical")

    return feature_types

## Building the Decision Tree

The `decision_tree()` function constructs a decision tree recursively by selecting the best feature, splitting the dataset, and building child subtrees.

### 1. Stopping Conditions

The recursion stops when one of these conditions is met:

- **Pure node:** All samples belong to the same class, so that class is returned.
- **No features remain:** The majority class is returned.
- **Maximum depth reached:** If `max_depth` is specified and the current depth reaches it, the majority class is returned.
- **Minimum sample condition:** If the number of samples is less than `min_split`, the majority class is returned.

### 2. Selecting the Best Feature

The `best_feature()` function evaluates the available features and returns the index of the feature with the highest information gain. If no feature is selected, the function returns the majority class.

### 3. Creating a Decision Node

Each internal node is represented by a dictionary containing:

- `feature`: The name of the selected feature.
- `majority`: The majority class at the current node, used as a fallback for unseen categorical values during prediction.
- `branches`: A dictionary that stores the child branches.
- `threshold`: The splitting threshold, included only for numerical features.

### 4. Splitting and Recursion

**Categorical features:**
- Create one branch for each unique category.
- Remove the selected feature from the feature matrix and feature names.
- Recursively build a subtree for each category.

**Numerical features:**
- Store the selected threshold.
- Divide samples into left and right branches.
- Remove the selected feature from the feature matrix and feature names.
- Recursively build a subtree for each branch.

### 5. Returning the Tree

The function returns a nested dictionary representing the tree. Internal nodes contain splitting information, while leaf nodes return predicted class labels.

This process continues recursively until a stopping condition is reached for every branch.

In [139]:
def decision_tree(X, y, feature_names, features_type ,depth = 0,max_depth = None,min_split = 2,criterion = "entropy"):

    # Stop if the node is pure
    if len(np.unique(y)) == 1:
        return y[0]

    #  Stop if no features remain
    if X.shape[1] == 0:
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]
    # checking max depth
    if max_depth is not None and depth>=max_depth:
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]
    # checking min split
    if len(y)<min_split:
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

    # Select the best feature
    best_feature_index= best_feature(X, y,features_type,criterion)
    if best_feature_index is not None:
        root_feature = best_feature_index
    else:
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]
    root_name = feature_names[root_feature]
    root_labels , root_counts = np.unique(y, return_counts=True)
    max_label = root_labels[np.argmax(root_counts)]

    #  Create branches
    subsets = split_dataset(X, y, root_feature,features_type[root_feature],criterion)
    features_type_rem = np.delete(features_type, root_feature).tolist()


    tree = {
        "feature" : root_name,
        "majority" : max_label,
        "branches": {}}

    #  Build each child
    remaining_X_names = np.delete(
        np.array(feature_names), root_feature
    ).tolist()
    if features_type[root_feature] == "categorical":
        for key, (X_sub,y_sub,(unique,counts)) in subsets.items():
            X_remaining = np.delete(X_sub, root_feature, axis=1)

            tree["branches"][key] = decision_tree(
                X_remaining, y_sub, remaining_X_names,features_type_rem,
                depth+1,max_depth,min_split,criterion
            )

    elif features_type[root_feature] == "numerical":
        tree["threshold"] = subsets["threshold"]
        subsets = subsets["subsets"]

        for key,(X_sub,y_sub) in subsets.items():
            X_remaining = np.delete(X_sub, root_feature, axis=1)
            tree["branches"][key] = decision_tree(X_remaining,y_sub,remaining_X_names,features_type_rem,depth+1,max_depth,min_split,criterion)


    return tree


##  Training the Decision Tree

In this step, we train our decision tree using the training dataset.

1. **Detect feature types:** Identify each feature as numerical or categorical.
2. **Build the tree:** Call `decision_tree()` with the training features, target labels, feature names, and feature types.
3. **Control tree growth:**
   - `max_depth=5` limits the maximum depth of the tree to 5.
   - `min_split=10` stops splitting a node when it contains fewer than 10 samples.

Finally, print the resulting tree structure to inspect its features, thresholds, branches, and leaf predictions.

In [140]:
feature_types = detect_feature_types(X_train)

tree = decision_tree(
    X_train,
    y_train,
    feature_names,
    feature_types,
    max_depth=5,
    min_split=10,
    criterion = "entropy"
)



##  Making Predictions

The `predict()` function predicts the class label of a sample by traversing the trained decision tree from the root node to a leaf node.

### How It Works

1. **Check for a leaf node:** If the current node is not a dictionary, return its class label.
2. **Identify the feature:** Find the index of the feature used by the current node.
3. **Follow the appropriate branch:**
   - **Numerical feature:** If the value is greater than the threshold, follow the right branch; otherwise, follow the left branch.
   - **Categorical feature:** Follow the branch corresponding to the feature's category.
4. **Handle unseen categories:** If a category has no corresponding branch, return the majority class stored at the current node.
5. **Continue recursively:** Repeat the process until a leaf node is reached.

The function returns the predicted class label for the given sample.

In [141]:
def predict(tree,sample,features,feature_type):
 # will return a value at that particular node cuz we will have only one node at this level
    if not isinstance(tree, dict):
        return tree


    feature_name = tree["feature"]
    feature_index = features.index(feature_name)
    feature_value = sample[feature_index]
    if feature_type[feature_index] == "numerical":
        threshold = tree["threshold"]
        if feature_value > threshold:
            next_branch = tree["branches"]["right"]
        elif feature_value <= threshold:
            next_branch = tree["branches"]["left"]
        # for categorical features
    else:
        if feature_value not in tree["branches"]:
            return tree["majority"]
        next_branch = tree["branches"][feature_value]





    return predict(next_branch,sample,features,feature_type)



## Generating Training Predictions

In this step, we generate predictions for all samples in the training dataset using our trained decision tree.

- Iterate through each row in `X_train`.
- Pass each sample to the `predict()` function along with the trained tree, feature names, and feature types.
- Collect the predicted labels into a NumPy array using `np.array()`.

The resulting `y_hat_train` array contains the predicted class label for every training sample. These predictions can be compared with `y_train` to calculate training accuracy and evaluate how well the model fits the training data.

In [142]:
y_hat_train = np.array([
    predict(tree, row, feature_names,feature_types)
    for row in X_train
])


##  Generating Predictions with Our Decision Tree

In this step, we generate predictions for all test samples using our from-scratch decision tree.

- Iterate through each row in `X_test`.
- Pass each sample to the `predict()` function along with the trained tree, feature names, and feature types.
- Collect all predicted labels into a NumPy array using `np.array()`.

The resulting `my_predictions` array contains the predicted class label for every test sample. These predictions can be used to calculate test accuracy, construct a confusion matrix, and compare our implementation with Scikit-learn.

In [143]:
y_hat_test = np.array([
    predict(tree, row, feature_names,feature_types)
    for row in X_test
])


##  Evaluating Training and Test Accuracy

In this step, we evaluate our from-scratch decision tree using training accuracy and test accuracy.

- **Training Accuracy:** Measures the proportion of training samples correctly classified by the model.
- **Test Accuracy:** Measures the proportion of unseen test samples correctly classified by the model.

### Formula

$$
\text{Accuracy} =
\frac{\text{Number of Correct Predictions}}
{\text{Total Number of Samples}}
$$

Comparing both accuracy scores helps us understand how well the model fits the training data and how effectively it generalizes to unseen samples.

- Similar training and test accuracy may indicate reasonable generalization.
- Much higher training accuracy than test accuracy may indicate overfitting.
- Low training and test accuracy may indicate underfitting.

Accuracy should be interpreted alongside other evaluation metrics, such as the confusion matrix, especially when the classes are imbalanced.

In [144]:
print("Training Accuracy : ", accuracy_score(y_hat_train, y_train))
print("Test Accuracy : ", accuracy_score(y_hat_test, y_test))


Training Accuracy :  0.8963133640552995
Test Accuracy :  0.8930875576036866


## Decision Tree Using Scikit-learn

In this step, we train a decision tree using Scikit-learn and evaluate its test accuracy to compare it with our from-scratch implementation.

### How It Works

1. **Initialize the encoder:** Create an `OrdinalEncoder` to convert categorical feature values into numerical representations.
2. **Fit and transform training data:** Learn the category mappings from `X_train` and encode its values.
3. **Transform test data:** Apply the same mappings to `X_test` to maintain consistency.
4. **Initialize the model:** Create a `DecisionTreeClassifier` with a fixed random state for reproducibility.
5. **Train the model:** Fit the classifier using the encoded training features and `y_train`.
6. **Make predictions:** Predict the labels for `X_test`.
7. **Evaluate performance:** Calculate test accuracy using Scikit-learn's `accuracy_score()`.

The resulting accuracy provides a baseline for comparing our custom decision tree with an established implementation.

In [145]:
encoder = OrdinalEncoder()

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

sk_model = DecisionTreeClassifier(random_state=42)
sk_model.fit(X_train_encoded, y_train)

sk_predictions = sk_model.predict(X_test_encoded)

print("Sklearn accuracy:", accuracy_score(y_test, sk_predictions))
print("From Scratch Accuracy: ", accuracy_score(y_test, y_hat_test))

Sklearn accuracy: 0.8930875576036866
From Scratch Accuracy:  0.8930875576036866


##  Comparing Confusion Matrices

A confusion matrix summarizes the classification results by comparing actual labels with predicted labels.

In this step, we compare the confusion matrices of our from-scratch decision tree and Scikit-learn's decision tree using the same test dataset.

Each confusion matrix contains:

- **True Negatives (TN):** Negative samples correctly predicted as negative.
- **False Positives (FP):** Negative samples incorrectly predicted as positive.
- **False Negatives (FN):** Positive samples incorrectly predicted as negative.
- **True Positives (TP):** Positive samples correctly predicted as positive.

For binary classification, the matrix has the following structure:

$$
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
$$

Comparing these matrices helps us understand which types of classification errors each model makes, beyond what accuracy alone reveals.

In [146]:
print("My Decision Tree:")
print(confusion_matrix(y_test, y_hat_test, labels=["No", "Yes"]))

print("\nScikit-learn Decision Tree:")
print(confusion_matrix(y_test, sk_predictions, labels=["No", "Yes"]))

My Decision Tree:
[[249 116]
 [  0 720]]

Scikit-learn Decision Tree:
[[249 116]
 [  0 720]]


##  Breast Cancer Dataset

In this step, we load the Breast Cancer dataset to test our from-scratch decision tree on a new classification problem.

Unlike the Play Tennis dataset, which predicts whether to play tennis based on weather conditions, this dataset is used for breast cancer classification.

We will explore the dataset, identify its features and target variable, preprocess the data if necessary, and evaluate our decision tree's classification performance.

In [147]:
df2 = pd.read_csv("./breast-cancer.csv")

In [148]:
df2.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [149]:
df2.isna().sum()

id                         0
diagnosis                  0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64

In [150]:
X = df2.drop(columns=["id","diagnosis"]).to_numpy()
y = df2["diagnosis"].to_numpy()
feature_names =  df2.drop(columns=["id","diagnosis"]).columns.to_list()


## Splitting Train and Test Data

In [151]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,train_size=0.8)


In [152]:
print("shape of X train",X_train.shape)
print("shape of Y train",y_train.shape)
print("shape of X_test",X_test.shape)
print("shape of y_test",y_test.shape)
X_train[0]

shape of X train (455, 30)
shape of Y train (455,)
shape of X_test (114, 30)
shape of y_test (114,)


array([1.719e+01, 2.207e+01, 1.116e+02, 9.283e+02, 9.726e-02, 8.995e-02,
       9.061e-02, 6.527e-02, 1.867e-01, 5.580e-02, 4.203e-01, 7.383e-01,
       2.819e+00, 4.542e+01, 4.493e-03, 1.206e-02, 2.048e-02, 9.875e-03,
       1.144e-02, 1.575e-03, 2.158e+01, 2.933e+01, 1.405e+02, 1.436e+03,
       1.558e-01, 2.567e-01, 3.889e-01, 1.984e-01, 3.216e-01, 7.570e-02])

## Detecting Feature Types

We use the `detect_feature_types()` function to identify whether each feature in the training dataset is numerical or categorical.

The function attempts to convert each column to floating-point values:

- **Numerical:** If conversion succeeds, the feature is classified as numerical.
- **Categorical:** If conversion raises a `ValueError` or `TypeError`, the feature is classified as categorical.

The resulting `feature_types` list determines how our decision tree handles each feature when selecting splits.

This step helps our implementation adapt to datasets containing different types of features, including the Breast Cancer dataset.

In [153]:
feature_types = detect_feature_types(X_train)


## Training the Decision Tree on the Breast Cancer Dataset

In this step, we train our from-scratch decision tree using the Breast Cancer training dataset.

The `decision_tree()` function recursively selects the best feature, splits the dataset, and builds child subtrees until a stopping condition is reached.

### Parameters

- `X_train`: Training features.
- `y_train`: Target labels for the training samples.
- `feature_names`: Names of the input features.
- `feature_types`: Detected types of the input features.
- `max_depth=5`: Limits the tree's maximum depth to 5.
- `min_split=2`: Stops splitting a node when it contains fewer than 2 samples.

The resulting `tree` contains the learned decision rules and leaf predictions. We can use it to make predictions and evaluate its performance on both training and test data.

In [154]:
tree = decision_tree(X_train, y_train, feature_names,feature_types,max_depth=5,min_split=2,criterion = "entropy")

In [155]:
y_hat = np.array([predict(tree,row, feature_names,feature_types)
                    for row in X_test])
print(f"From Scratch Accuracy : {(y_test == y_hat).sum() / len(y_test)}")

From Scratch Accuracy : 0.9385964912280702


##  Training a Scikit-learn Decision Tree

In this step, we train Scikit-learn's `DecisionTreeClassifier` on the Breast Cancer dataset to compare it with our from-scratch implementation.

### Parameters

- `max_depth=5`: Limits the maximum depth of the tree to 5.
- `min_samples_split=2`: A node must contain at least 2 samples to be eligible for splitting.
- `random_state=42`: Ensures reproducibility where randomness affects the algorithm.

### How It Works

1. Initialize the decision tree with the specified parameters.
2. Train the model using `X_train` and `y_train`.
3. Predict the target labels for `X_test` using the trained model.

The resulting `sk_pred` array contains the predicted labels for the test samples.

**Note:** These settings make the tree-growth limits comparable to our custom implementation, but the models may still differ in split criteria and feature-handling strategies.

In [156]:
sk = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=2,
    random_state=42
)

sk.fit(X_train, y_train)
sk_pred = sk.predict(X_test)

##  Evaluating and Comparing Both Decision Trees

In this step, we compare the performance of our from-scratch decision tree with Scikit-learn's implementation on the Breast Cancer test dataset.

### 1. Accuracy

Accuracy measures the proportion of test samples classified correctly.

We calculate accuracy for both models to compare their overall classification performance.

### 2. Confusion Matrix

A confusion matrix compares the actual labels with the predicted labels.

For this dataset, the labels are:
- **B:** Benign
- **M:** Malignant

With the label order `["B", "M"]`, the confusion matrix is structured as:

$$
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
$$

Here, the matrix shows how many benign and malignant cases each model classifies correctly or incorrectly.

### 3. Comparing Results

Comparing the accuracy scores shows the overall performance of both models, while their confusion matrices reveal the types of errors they make.

Differences in results may arise because the two implementations use different splitting strategies, impurity calculations, or feature-handling methods.

In [157]:
print(f"From Scratch Accuracy : {(y_test == y_hat).sum() / len(y_test)}")
print(f"Sklearn Accuracy : {accuracy_score(y_test, sk_pred)}")
print("From Scratch Confusion Matrix : ",confusion_matrix(y_test, y_hat, labels=["B", "M"]))
print("SKlearn Confusion Matrix: ", confusion_matrix(y_test, sk_pred, labels=["B", "M"]))

From Scratch Accuracy : 0.9385964912280702
Sklearn Accuracy : 0.9210526315789473
From Scratch Confusion Matrix :  [[68  0]
 [ 7 39]]
SKlearn Confusion Matrix:  [[67  1]
 [ 8 38]]


##  Titanic Dataset: Handling Missing Values

In this step, we load the Titanic training dataset and inspect it for missing values.

### How It Works

1. **Load the dataset:** Read `train.csv` into a Pandas DataFrame.
2. **Check missing values:** Use `isna().sum()` to count missing values in each column.
3. **Remove incomplete rows:** Use `dropna()` to remove rows containing one or more missing values.
4. **Verify the result:** Run `isna().sum()` again to confirm that no missing values remain.

The Titanic dataset contains both numerical and categorical features, making it useful for testing our decision tree's ability to handle mixed data types.

**Note:** Dropping every row with missing values can discard useful data and reduce the dataset size. For a more robust approach, consider imputing missing values, especially in important features such as `Age` and `Embarked`.

In [180]:
df3 = pd.read_csv("train.csv")
df3.isna().sum()
df3 = df3.dropna()
df3.isna().sum()


PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64

In [181]:
df3.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

##  Feature Selection and Train-Test Split

In this step, we select the input features and target variable from the Titanic dataset and divide the data into training and testing sets.

### 1. Feature Selection

We use four features to predict passenger survival:

- `Age`: Passenger's age (numerical).
- `Fare`: Ticket fare (numerical).
- `Sex`: Passenger's sex (categorical).
- `Embarked`: Port where the passenger boarded (categorical).

The target variable is `Survived`, which indicates whether a passenger survived.

We convert the selected features into a NumPy array with `dtype=object` to preserve both numerical and categorical values.

### 2. Train-Test Split

We divide the dataset using `train_test_split()`:

- `test_size=0.2`: Allocates 20% of the data for testing and 80% for training.
- `random_state=42`: Ensures a reproducible split.
- `stratify=y`: Maintains approximately the same class distribution in both sets.

This split allows us to train the decision tree on one portion of the data and evaluate its ability to predict survival for unseen passengers.

In [182]:
features = ["Age", "Fare", "Sex", "Embarked"]

X = df3[features].to_numpy(dtype=object)
y = df3["Survived"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

## Training the Decision Tree on the Titanic Dataset

In this step, we detect the feature types and train our from-scratch decision tree to predict passenger survival.

### How It Works

1. **Detect feature types:** The `detect_feature_types()` function identifies numerical features (`Age`, `Fare`) and categorical features (`Sex`, `Embarked`).
2. **Build the tree:** The `decision_tree()` function recursively selects features, creates branches, and builds subtrees.
3. **Control tree growth:**
   - `max_depth=5` limits the maximum tree depth to 5.
   - `min_split=2` stops splitting when a node contains fewer than 2 samples.

The resulting `tree` contains the learned decision rules for predicting whether a passenger survived.

This dataset tests our implementation's ability to handle mixed feature types and numerical threshold-based as well as categorical splits.

In [183]:
feature_types = detect_feature_types(X_train)

tree = decision_tree(X_train,y_train,features,feature_types,max_depth=5,min_split=2,criterion = "entropy")

In [184]:
y_hat = np.array([predict(tree,row,features,feature_types) for row in X_test])

## Training a Scikit-learn Decision Tree on the Titanic Dataset

In this step, we prepare the mixed-type features for Scikit-learn and train a decision tree to predict passenger survival.

### How It Works

1. **Copy the datasets:** Create copies of the training and testing features so the original arrays remain unchanged.
2. **Encode categorical features:** Use `OrdinalEncoder` to convert `Sex` and `Embarked` into numerical values.
3. **Handle unseen categories:** Set `handle_unknown="use_encoded_value"` and `unknown_value=-1` so previously unseen categories in the test set receive the value `-1`.
4. **Convert to numeric:** Convert all feature values to `float` so the model can process them.
5. **Train the model:** Fit a `DecisionTreeClassifier` using the Gini impurity criterion, a maximum depth of 5, and a minimum of 2 samples required to split a node.
6. **Generate predictions:** Use the trained model to predict survival outcomes for the test set.

### Why Use Scikit-learn?

This model provides a baseline for comparing our from-scratch decision tree against a well-established implementation.

Using the same `max_depth=5` and `min_samples_split=2` settings makes the comparison more meaningful, although the models may still produce different trees because of differences in split selection and categorical-feature handling.

In [185]:
# Copy the data so the original arrays remain unchanged
X_train_sk = X_train.copy()
X_test_sk = X_test.copy()

# Categorical column indices: Sex = 2, Embarked = 3
categorical_indices = [2, 3]

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_sk[:, categorical_indices] = encoder.fit_transform(
    X_train_sk[:, categorical_indices]
)

X_test_sk[:, categorical_indices] = encoder.transform(
    X_test_sk[:, categorical_indices]
)

# Convert the complete feature arrays to numeric
X_train_sk = X_train_sk.astype(float)
X_test_sk = X_test_sk.astype(float)

# Train sklearn's decision tree
sk_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    min_samples_split=2,
)

sk_model.fit(X_train_sk, y_train)

sk_pred = sk_model.predict(X_test_sk)



##  Comparing Decision Tree Performance on the Titanic Dataset

In this step, we evaluate both decision tree implementations on the same test set and compare their predictions.

### Evaluation Metrics

1. **Accuracy:** Measures the proportion of passengers whose survival outcomes were predicted correctly.
2. **Confusion Matrix:** Shows the number of correct and incorrect predictions for each survival class.
3. **Identical Predictions:** Checks whether both models predict exactly the same outcome for every test sample.
4. **Different Predictions:** Counts how many test samples receive different predictions from the two models.

### Why Compare Both Models?

This comparison helps us understand how our from-scratch implementation performs relative to Scikit-learn's decision tree.

Even when both models achieve similar accuracy, their individual predictions may differ because their splitting strategies and handling of categorical features are different.

In [187]:
print("From SCratch Accuracy:", accuracy_score(y_test, y_hat))
print("from Scratch Confusion Matrix :\n ",confusion_matrix(y_test, y_hat))
print("Sklearn accuracy:", accuracy_score(y_test, sk_pred))
print("Skleanr Confusion Matrix :\n ",confusion_matrix(y_test, sk_pred))



From SCratch Accuracy: 0.7297297297297297
from Scratch Confusion Matrix :
  [[13  1]
 [ 9 14]]
Sklearn accuracy: 0.7297297297297297
Skleanr Confusion Matrix :
  [[11  3]
 [ 7 16]]


# Conclusion: Decision Tree Implementation from Scratch

In this project, I implemented a Decision Tree Classifier from scratch using Python, NumPy, and Pandas, and compared its performance with Scikit-learn's implementation across three datasets.

## Key Implementations

- **Entropy and Gini Impurity:** Implemented impurity measures to evaluate the quality of splits.
- **Dataset Splitting:** Developed logic for categorical features and numerical threshold-based splits.
- **Information Gain:** Calculated impurity reduction to evaluate candidate features.
- **Best Feature Selection:** Selected the feature providing the highest information gain.
- **Recursive Tree Construction:** Built a decision tree recursively with configurable maximum depth and minimum split size.
- **Prediction:** Implemented traversal of the learned tree to predict class labels.
- **Evaluation:** Compared accuracy, confusion matrices, and predictions against Scikit-learn.

## Datasets Used

1. **Play Tennis:** Tested the implementation on a categorical dataset.
2. **Breast Cancer:** Evaluated the classifier on a numerical dataset.
3. **Titanic:** Tested the implementation on mixed numerical and categorical features.

## Key Learnings

- Understood how decision trees select features and recursively partition datasets.
- Learned to handle categorical and numerical features using different splitting strategies.
- Explored how hyperparameters influence tree growth and model performance.
- Compared a custom implementation with a standard machine learning library.
- Practiced data preprocessing, train-test splitting, prediction, and model evaluation.

## Final Reflection

Implementing a decision tree from scratch strengthened my understanding of the algorithm beyond simply using a machine learning library. Comparing it with Scikit-learn also helped me understand how differences in preprocessing, split selection, and categorical-feature handling can affect predictions and performance.

This project provided practical experience in translating machine learning concepts into working code and evaluating models across different types of datasets.